[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S09_pandas_series_dataframe.ipynb)

# Sesión 09 · pandas: `Series` y `DataFrame`

**Módulo 3: Pandas** · ⏱️ Duración estimada: 60 minutos (más el avance del proyecto, que es trabajo aparte)

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Crear una `Series` y un `DataFrame`, y explicar el papel del índice.
2. Cargar un CSV con `pd.read_csv` (también con otro separador) y explorarlo con `head`, `info`, `describe`, `shape` y `dtypes`.
3. Seleccionar columnas, y filas por etiqueta con `loc` o por posición con `iloc`.
4. Filtrar filas con condiciones y con `query`.

## 📋 Qué debes saber antes
Módulos 1 y 2: listas, diccionarios, funciones y máscaras booleanas de NumPy.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Al final está el 🧱 **Avance del proyecto** (P2, parte 1): se hace en tu propio repositorio.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Importa pandas como `pd`, genera los datos de práctica, escribe dos archivos (`ventas.csv`, separado por comas, y `movimientos.csv`, separado por punto y coma) y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, escribe dos archivos CSV de práctica y carga los verificadores.
import copy
import csv
import hashlib
import io
import math
import statistics

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
ventas_semana = [int(v) for v in rng.integers(800, 5000, size=7)]
inv_productos = ["polo", "jean", "casaca", "gorra", "medias"]
inv_precios = [round(float(x), 2) for x in rng.uniform(15, 200, size=5)]
inv_stock = [int(x) for x in rng.integers(0, 50, size=5)]

_CATALOGO = [("polo básico", "polo", 39.9), ("polo piqué", "polo", 59.9), ("jean slim", "jean", 129.9),
             ("jean mom", "jean", 149.9), ("casaca denim", "casaca", 189.9), ("casaca polar", "casaca", 169.9),
             ("gorra", "accesorio", 25.0)]
_TIENDAS = ["Miraflores", "Surco", "Lince", "Barranco", "San Isidro"]
_COLS_V = ["id_venta", "fecha", "tienda", "producto", "categoria", "unidades", "precio", "canal"]
_filas_v = []
for _i in range(60):
    _prod = _CATALOGO[int(rng.integers(0, len(_CATALOGO)))]
    _filas_v.append([1001 + _i, f"2026-09-{int(rng.integers(1, 31)):02d}", str(rng.choice(_TIENDAS)), _prod[0], _prod[1],
                     int(rng.integers(1, 9)), _prod[2], str(rng.choice(["app", "tienda", "web"]))])
_buf = io.StringIO()
csv.writer(_buf, lineterminator="\n").writerows([_COLS_V] + _filas_v)
_TXT_V = _buf.getvalue()
with open("ventas.csv", "w", encoding="utf-8") as _f:
    _f.write(_TXT_V)

# ---------- Datos de práctica: movimientos bancarios (separados por punto y coma) ----------
_COLS_M = ["id_mov", "fecha", "cliente", "tipo", "canal", "monto"]
_filas_m = []
for _i in range(50):
    _tipo = str(rng.choice(["deposito", "retiro", "pago", "transferencia"], p=[0.25, 0.35, 0.25, 0.15]))
    _canal = str(rng.choice(["app", "agencia", "cajero"])) if _tipo == "retiro" else str(rng.choice(["app", "agencia"]))
    _monto = rng.uniform(100, 4000) if _tipo == "deposito" else -rng.uniform(20, 1200)
    _filas_m.append([2001 + _i, f"2026-09-{int(rng.integers(1, 31)):02d}", f"C{int(rng.integers(1, 13)):02d}",
                     _tipo, _canal, f"{_monto:.2f}"])
_buf = io.StringIO()
csv.writer(_buf, delimiter=";", lineterminator="\n").writerows([_COLS_M] + _filas_m)
_TXT_M = _buf.getvalue()
with open("movimientos.csv", "w", encoding="utf-8") as _f:
    _f.write(_TXT_M)

_D = copy.deepcopy({k: globals()[k] for k in ["dias", "ventas_semana", "inv_productos", "inv_precios", "inv_stock"]})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _leer(txt, sep):
    """Referencia con el módulo csv (no con pandas)."""
    filas = list(csv.reader(io.StringIO(txt), delimiter=sep))
    salida = []
    for fila in filas[1:]:
        convertida = []
        for x in fila:
            try:
                convertida.append(int(x))
            except ValueError:
                try:
                    convertida.append(float(x))
                except ValueError:
                    convertida.append(x)
        salida.append(convertida)
    return filas[0], salida


_CV, _V = _leer(_TXT_V, ",")
_CM, _M = _leer(_TXT_M, ";")


def _cols(filas, cols_origen, cols):
    idx = [cols_origen.index(c) for c in cols]
    return [[f[i] for i in idx] for f in filas]


def _donde(filas, cond):
    return [i for i, f in enumerate(filas) if cond(f)]


def _es_texto(a, b):
    return isinstance(a, str) and str(a) == b


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    d, v = _D["dias"], _D["ventas_semana"]
    _ser(r, "ventas_dia", v, "los valores son `ventas_semana` y las etiquetas, `dias`", indice=d)
    _esc(r, "venta_jueves", v[3], "búscala por su etiqueta")
    _ser(r, "con_igv", [round(x * 118 / 100, 2) for x in v], "cada venta por 1.18, redondeada a 2 decimales", indice=d, tol=0.0051)
    arriba = [i for i, x in enumerate(v) if x >= 3000]
    _ser(r, "sobre_meta", [v[i] for i in arriba], "solo los días con venta de 3000 o más, con su etiqueta", indice=[d[i] for i in arriba])
    _esc(r, "total", math.fsum(v), "suma todas las ventas")
    r.valor("mejor_dia", d[v.index(max(v))], None, "debería ser la etiqueta del día con mayor venta", igual=_es_texto)
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_posicion": "0e939e9ca064726168df510a5b86d75df221609e471cb3d8c10019b5aff835d1",
        "pred_alineada": "60e92dc9a63f3fe589e17c2542aea18087ca7a6dc94926ad987d12fd721f9c20",
        "pred_sin_pareja": "7cd2a33d8476047b3755295f8b4747a3baea572e22f0e24d21505be7bc3c9844",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2")
    p, pr, st = _D["inv_productos"], _D["inv_precios"], _D["inv_stock"]
    _df(r, "inventario", ["producto", "precio", "stock"], [list(t) for t in zip(p, pr, st)],
        "cada columna debería salir de su lista, en el orden producto, precio, stock")
    _esc(r, "n_filas", len(p), "cuenta las filas")
    _esc(r, "n_columnas", 3, "cuenta las columnas")
    r.valor("columnas", ["producto", "precio", "stock"], list, "convierte las columnas en lista con `list()`")
    _ser(r, "stock_s", st, "debería ser la columna stock")
    _esc(r, "valor_inventario", round(math.fsum(a * b for a, b in zip(pr, st)), 2),
         "precio por stock de cada producto, sumado y redondeado a 2 decimales", tol=0.0051)
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    _df(r, "ventas", _CV, _V, "carga `ventas.csv` con `pd.read_csv`")
    _df(r, "movs", _CM, _M, "carga `movimientos.csv` indicando el separador")
    _df(r, "primeras", _CV, _V[:3], "las 3 primeras filas de `ventas`", indice=[0, 1, 2])
    _df(r, "ultimas", _CM, _M[-2:], "las 2 últimas filas de `movs`", indice=[48, 49])
    r.valor("forma_movs", (len(_M), len(_CM)), tuple, "usa el atributo que da (filas, columnas)")
    r.valor("tipo_unidades", "int64", str, "convierte a texto el dtype de la columna unidades")
    res = r.var("resumen")
    if res is not _FALTA:
        numericas = [c for c, x in zip(_CV, _V[0]) if isinstance(x, (int, float))]
        if not isinstance(res, pd.DataFrame):
            r.mal("`resumen` debería ser el DataFrame que devuelve `describe()`.")
        elif [str(c) for c in res.columns] != numericas or "mean" not in res.index:
            r.mal(f"`resumen` debería tener una columna por cada columna numérica de `ventas` ({numericas}) y una fila `mean`.")
        elif abs(float(res.loc["mean", "precio"]) - statistics.fmean([f[6] for f in _V])) > 1e-6:
            r.mal("La media del precio en `resumen` no coincide: ¿aplicaste `describe()` a `ventas`?")
        else:
            r.ok("`resumen` es correcto.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_forma_sin_sep": "0b2c226889380834076478d9ac9f71f17322e55f02a993c3a31cc11dd2419864",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    _ser(r, "precios_s", [f[6] for f in _V], "debería ser la columna precio de `ventas`")
    cols = ["tienda", "producto", "unidades"]
    _df(r, "sub", cols, _cols(_V, _CV, cols), "las columnas tienda, producto y unidades, en ese orden")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_tipo_1": "23d607e40e008b164500ee2a5fe34709292ea643d7fe03747c2fb98193cd8823",
        "pred_tipo_2": "d3173d3567944c6800b530a5a8c6cec37c69d038e101b2ef79ab221a2b8a61cb",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    _ser(r, "primera_venta", _V[0], "debería ser la primera fila de `ventas`", indice=_CV)
    _df(r, "bloque", _CV[:3], [f[:3] for f in _V[:5]], "las 5 primeras filas y las 3 primeras columnas, por posición")
    _esc(r, "ultimo_precio", _V[-1][6], "el precio de la última fila")
    _ser(r, "fila_10", _V[10], "la fila cuya etiqueta es 10", indice=_CV)
    _df(r, "rango_loc", ["tienda", "producto"], _cols(_V[5:9], _CV, ["tienda", "producto"]),
        "con `loc`, las filas de etiqueta 5 a 8 (el final se incluye)", indice=[5, 6, 7, 8])
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_loc_filas": "345e42ad061bc48d0359a369b1c6c567a1c82108b53e556c40e79578d7443105",
        "pred_iloc_filas": "7a20ad6fbd72c40ec9ce42fcce75b2afd01e02ef19725a1956c8b5a8b66eeefc",
    })
    r.fin()


def check_ejercicio_6():
    r = _Revision("Ejercicio 6")
    t, u, cat, pr, ca = (_CV.index(c) for c in ("tienda", "unidades", "categoria", "precio", "canal"))
    for nombre, cond, pista in (
        ("app", lambda f: f[ca] == "app", "las ventas del canal app"),
        ("grandes_surco", lambda f: f[t] == "Surco" and f[u] >= 5, "ventas de Surco con 5 unidades o más, las dos condiciones a la vez"),
        ("jean_o_casaca", lambda f: f[cat] in ("jean", "casaca"), "las ventas de categoría jean o casaca"),
        ("caras_tienda", lambda f: f[pr] > 150 and f[ca] == "tienda", "precio mayor que 150 y canal tienda"),
        ("sin_resultados", lambda f: f[u] > 100, "un filtro de unidades mayores que 100 (no hay ninguna)"),
    ):
        idx = _donde(_V, cond)
        _df(r, nombre, _CV, [_V[i] for i in idx], pista, indice=idx)
    idx = _donde(_V, lambda f: f[pr] > 150)
    _ser(r, "productos_caros", [_V[i][3] for i in idx], "solo la columna producto de las ventas con precio mayor que 150", indice=idx)
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    cl, ti, ca, mo, idm = (_CM.index(c) for c in ("cliente", "tipo", "canal", "monto", "id_mov"))
    _esc(r, "n_clientes", len({f[cl] for f in _M}), "cuenta los clientes distintos")
    idx = _donde(_M, lambda f: f[ti] == "retiro" and f[ca] == "app")
    _df(r, "retiros_app", _CM, [_M[i] for i in idx], "los movimientos de tipo retiro hechos por app", indice=idx)
    _esc(r, "total_retiros_app", round(math.fsum(_M[i][mo] for i in idx), 2), "suma el monto de esos retiros (será negativo) y redondea a 2 decimales", tol=0.0051)
    montos = [f[mo] for f in _M]
    _ser(r, "mov_mayor", _M[montos.index(max(montos))], "debería ser la fila completa del movimiento de mayor monto", indice=_CM)
    grandes = sorted({f[cl] for f in _M if f[ti] == "deposito" and f[mo] > 2000})
    r.valor("clientes_grandes", grandes, list, "lista ordenada de los clientes con algún depósito mayor que 2000, sin repetir")
    _esc(r, "pct_negativos", round(len([x for x in montos if x < 0]) * 100 / len(montos), 1), "porcentaje de montos negativos, con 1 decimal", tol=0.051)
    res = r.var("resumen_monto")
    if res is not _FALTA:
        if not isinstance(res, pd.Series) or "mean" not in res.index or "count" not in res.index:
            r.mal("`resumen_monto` debería ser la Series que devuelve `describe()` sobre la columna monto.")
        elif int(res["count"]) != len(montos) or abs(float(res["mean"]) - statistics.fmean(montos)) > 1e-6:
            r.mal("`resumen_monto` no coincide con el resumen de la columna monto de `movs`.")
        else:
            r.ok("`resumen_monto` es correcto.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    cols = ["fecha", "tienda", "unidades"]
    _df(r, "mini", cols, _cols(_V[:10], _CV, cols), "solo esas tres columnas y las 10 primeras filas, desde `read_csv`")
    otras = [c for c in _CV if c != "id_venta"]
    _df(r, "por_id", otras, _cols(_V, _CV, otras), "el índice debería ser la columna id_venta", indice=[f[0] for f in _V])
    fila = next(f for f in _V if f[0] == 1025)
    _ser(r, "venta_1025", _cols([fila], _CV, otras)[0], "la fila cuya etiqueta de índice es 1025", indice=otras)
    r.fin()


print("✅ Setup listo. Datos generados, archivos CSV escritos y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print("🏪 Ventas de tiendas")
print("dias          =", dias)
print("ventas_semana =", ventas_semana)
print("inv_productos =", inv_productos)
print("inv_precios   =", inv_precios)
print("inv_stock     =", inv_stock)
print()
print("📄 Primeras líneas de los archivos:")
for archivo in ["ventas.csv", "movimientos.csv"]:
    with open(archivo, encoding="utf-8") as f:
        print(f"--- {archivo}")
        for _ in range(4):
            print(f.readline().rstrip())

---
## 1. `Series`: valores con etiquetas

### 📘 Concepto
pandas se importa con el alias `pd` (el setup ya lo hizo: `import pandas as pd`). Una **`Series`** es una columna de datos con un **índice**: una etiqueta para cada valor.

```python
s = pd.Series(valores, index=etiquetas)
```

- `s["etiqueta"]` busca por etiqueta; `s.iloc[0]` busca por posición.
- Las operaciones son vectorizadas, como en NumPy: `s * 1.18`, `s[s > 100]`, `s.sum()`, `s.mean()`.
- `s.idxmax()` y `s.idxmin()` devuelven la **etiqueta** del máximo y del mínimo.
- Al operar dos Series, pandas las **alinea por etiqueta**, no por posición. Las etiquetas que solo están en una de las dos dan `NaN` (valor faltante).

In [ ]:
visitas_ej = pd.Series([120, 340, 90], index=["lun", "mar", "mié"])
print(visitas_ej)
print(visitas_ej["mar"], visitas_ej.iloc[0])
print(visitas_ej[visitas_ej > 100])
print(visitas_ej.sum(), visitas_ej.idxmax())

otra_ej = pd.Series([10, 20], index=["mar", "jue"])
print(visitas_ej + otra_ej)        # se alinean por etiqueta

### ✍️ Tu turno · Ejercicio 1: la semana como Series
**Parte A.**
1. `ventas_dia`: una Series con los valores de `ventas_semana` y las etiquetas de `dias`.
2. `venta_jueves`: la venta del jueves, buscada por su etiqueta.
3. `con_igv`: las ventas con 18 % de IGV, redondeadas a 2 decimales (usa el método `.round(2)`).
4. `sobre_meta`: solo los días con venta de 3000 o más.
5. `total`: la suma de la semana, y `mejor_dia`: la **etiqueta** del día con mayor venta.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_posicion` | `pd.Series([10, 20, 30], index=["a", "b", "c"]).iloc[0]` | número |
| `pred_alineada` | `(pd.Series([1, 2], index=["a", "b"]) + pd.Series([10, 20], index=["b", "c"]))["b"]` | número o `"nan"` |
| `pred_sin_pareja` | la misma suma, pero en la etiqueta `"a"` | número o `"nan"` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`pd.Series(ventas_semana, index=dias)`. Después, todo se hace sobre `ventas_dia`.
</details>

<details><summary>💡 Pista 2</summary>

`sobre_meta` es un filtro con una máscara: `ventas_dia[ventas_dia >= 3000]`. Para `mejor_dia` usa `idxmax`, no `max`.
</details>

---
## 2. `DataFrame`: una tabla con columnas

### 📘 Concepto
Un **`DataFrame`** es una tabla: cada columna es una Series y todas comparten el mismo índice de filas. La forma más directa de crearlo es desde un diccionario `nombre_columna → lista de valores`:

```python
df = pd.DataFrame({"producto": [...], "precio": [...]})
```

- `df.shape` es `(filas, columnas)`; `df.columns`, los nombres de las columnas.
- `df["precio"]` devuelve una columna como Series.
- Las operaciones entre columnas se hacen fila a fila: `df["precio"] * df["stock"]`.
- Si no das un índice, pandas numera las filas desde 0.

In [ ]:
pedido_ej = pd.DataFrame({
    "producto": ["polo", "jean", "gorra"],
    "precio": [39.9, 129.9, 25.0],
    "unidades": [2, 1, 3],
})
print(pedido_ej)
print(pedido_ej.shape, list(pedido_ej.columns))
print((pedido_ej["precio"] * pedido_ej["unidades"]).sum())

### ✍️ Tu turno · Ejercicio 2: el inventario
1. `inventario`: un DataFrame con las columnas `producto`, `precio` y `stock`, en ese orden, a partir de `inv_productos`, `inv_precios` e `inv_stock`.
2. `n_filas` y `n_columnas`: su forma, desempaquetada en una línea.
3. `columnas`: los nombres de las columnas, como lista.
4. `stock_s`: la columna stock.
5. `valor_inventario`: la suma de precio por stock de todos los productos, redondeada a 2 decimales.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Las claves del diccionario son los nombres de las columnas y los valores, las listas del setup.
</details>

<details><summary>💡 Pista 2</summary>

`n_filas, n_columnas = inventario.shape`. Para el valor: multiplica las dos columnas, suma con `.sum()` y redondea con `round(..., 2)`.
</details>

---
## 3. Cargar un CSV y explorarlo

### 📘 Concepto
`pd.read_csv(ruta)` carga un archivo CSV en un DataFrame y detecta el tipo de cada columna. Si el archivo usa otro separador (en Perú y otros países se usa mucho el punto y coma, porque la coma es el separador decimal), indícalo con `sep=";"`.

Lo primero que se hace con un dataset nuevo:

| Código | Qué muestra |
|---|---|
| `df.head(n)` / `df.tail(n)` | las primeras / últimas `n` filas (5 por defecto) |
| `df.shape` | (filas, columnas) |
| `df.dtypes` | el tipo de cada columna (`int64`, `float64`, texto...) |
| `df.info()` | columnas, tipos y cuántos valores no nulos hay; **imprime**, no devuelve nada |
| `df.describe()` | resumen estadístico (conteo, media, desviación, mínimo, cuartiles, máximo) de las columnas numéricas |

In [ ]:
with open("mini_ej.csv", "w") as f:
    f.write("tienda;visitas;ventas\nSurco;120;3400.5\nLince;95;2890.0\nBarranco;140;4100.2\n")

mini_ej = pd.read_csv("mini_ej.csv", sep=";")
print(mini_ej.head(2))
print(mini_ej.shape)
print(mini_ej.dtypes)
mini_ej.info()
mini_ej.describe()

### ✍️ Tu turno · Ejercicio 3: cargar los dos archivos
**Parte A.**
1. `ventas`: carga `ventas.csv`.
2. `movs`: carga `movimientos.csv` (mira su separador en las primeras líneas que imprimió la celda de datos).
3. `primeras`: las 3 primeras filas de `ventas`, y `ultimas`: las 2 últimas de `movs`.
4. `forma_movs`: la forma de `movs`.
5. `tipo_unidades`: el tipo de la columna unidades de `ventas`, convertido a texto con `str()`.
6. `resumen`: el resumen estadístico de `ventas`.

Ejecuta también `ventas.info()` y míralo con calma.

**Parte B.** Predice **sin ejecutar**: `pred_forma_sin_sep` = `pd.read_csv("movimientos.csv").shape`, es decir, sin indicar el separador (una tupla).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Solo `movs` necesita `sep=";"`. `head` y `tail` reciben la cantidad de filas.
</details>

<details><summary>💡 Pista 2</summary>

Para `tipo_unidades`: `str(ventas["unidades"].dtype)`. En la parte B piensa qué hace pandas si no encuentra comas: ¿cuántas columnas ve?
</details>

---
## 4. Seleccionar columnas

### 📘 Concepto
- `df["col"]`, con **un** nombre, devuelve una **Series**.
- `df[["col1", "col2"]]`, con una **lista** de nombres (doble corchete), devuelve un **DataFrame** con esas columnas, en el orden de la lista.

Evita `df.col` (con punto): falla si el nombre tiene espacios o coincide con un método de pandas.

In [ ]:
pedido_ej = pd.DataFrame({"producto": ["polo", "jean"], "precio": [39.9, 129.9], "unidades": [2, 1]})
print(type(pedido_ej["precio"]))
print(pedido_ej[["unidades", "producto"]])

### ✍️ Tu turno · Ejercicio 4: columnas de ventas
**Parte A.**
1. `precios_s`: la columna precio de `ventas`.
2. `sub`: un DataFrame con las columnas tienda, producto y unidades, en ese orden.

**Parte B.** Predice **sin ejecutar** el nombre del tipo, como texto:

| Variable | Pregunta |
|---|---|
| `pred_tipo_1` | `type(ventas["precio"]).__name__` |
| `pred_tipo_2` | `type(ventas[["precio"]]).__name__` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Un corchete con un nombre, o dos corchetes con una lista de nombres.
</details>

<details><summary>💡 Pista 2</summary>

`sub = ventas[["tienda", "producto", "unidades"]]`. En la parte B, fíjate en cuántos corchetes hay.
</details>

---
## 5. Filas por etiqueta y por posición: `loc` e `iloc`

### 📘 Concepto
| | `loc` | `iloc` |
|---|---|---|
| Busca por | **etiqueta** (índice y nombres de columna) | **posición** (números desde 0) |
| Ejemplo | `df.loc[10, "precio"]` | `df.iloc[10, 6]` |
| Slicing `a:b` | **incluye** `b` | **excluye** `b` (como en Python) |

Las dos aceptan `[filas, columnas]`. Si das solo filas, devuelven todas las columnas. Una sola fila sale como Series (con los nombres de columna como etiquetas); varias filas, como DataFrame.

Con el índice por defecto (0, 1, 2...) las etiquetas y las posiciones coinciden, pero después de filtrar u ordenar ya no: por eso conviene tener clara la diferencia desde ahora.

In [ ]:
pedido_ej = pd.DataFrame({"producto": ["polo", "jean", "gorra", "casaca"],
                          "precio": [39.9, 129.9, 25.0, 189.9]})
print(pedido_ej.iloc[0])                  # primera fila, como Series
print(pedido_ej.iloc[1:3])                # posiciones 1 y 2
print(pedido_ej.loc[1:3, ["producto"]])   # etiquetas 1, 2 y 3
print(pedido_ej.loc[2, "precio"], pedido_ej.iloc[-1, 1])

### ✍️ Tu turno · Ejercicio 5: ubicar ventas
**Parte A.** Con `ventas`:
1. `primera_venta`: la primera fila, con `iloc`.
2. `bloque`: las 5 primeras filas y las 3 primeras columnas, con `iloc`.
3. `ultimo_precio`: el precio de la última fila, con `iloc` (el precio es la columna en la posición 6).
4. `fila_10`: la fila con etiqueta 10, con `loc`.
5. `rango_loc`: las filas de etiqueta 5 a 8 (ambas incluidas), solo con las columnas tienda y producto, con `loc`.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_loc_filas` | `len(ventas.loc[0:4])` |
| `pred_iloc_filas` | `len(ventas.iloc[0:4])` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

`iloc` usa números para filas y columnas; `loc` usa etiquetas de fila y nombres de columna.
</details>

<details><summary>💡 Pista 2</summary>

`rango_loc = ventas.loc[5:8, ["tienda", "producto"]]`. Para la última fila con `iloc`, usa `-1`.
</details>

---
## 6. Filtrar filas: máscaras y `query`

### 📘 Concepto
Igual que en NumPy: una condición sobre una columna da una máscara, y `df[mascara]` devuelve las filas que cumplen, **conservando sus etiquetas originales**.

- Varias condiciones: `&`, `|` y `~`, cada una entre paréntesis.
- `df["col"].isin([a, b])`: `True` si el valor está en la lista.
- `df.loc[mascara, "col"]`: filtra filas y elige columnas a la vez.
- `df.query("precio > 100 and canal == 'app'")`: la misma idea escrita como texto, más legible. Dentro de `query` se usan `and`, `or` y `not`, y los textos van entre comillas simples.

Si ninguna fila cumple, obtienes un DataFrame vacío con todas sus columnas.

In [ ]:
pedido_ej = pd.DataFrame({"producto": ["polo", "jean", "gorra", "casaca"],
                          "precio": [39.9, 129.9, 25.0, 189.9],
                          "canal": ["app", "web", "app", "tienda"]})
print(pedido_ej[pedido_ej["precio"] > 100])
print(pedido_ej[(pedido_ej["canal"] == "app") & (pedido_ej["precio"] < 30)])
print(pedido_ej[pedido_ej["producto"].isin(["jean", "gorra"])])
print(pedido_ej.query("precio > 30 and canal != 'web'"))
print(pedido_ej.loc[pedido_ej["canal"] == "app", "producto"])

### ✍️ Tu turno · Ejercicio 6: filtros sobre ventas
Con `ventas`:
1. `app`: las ventas del canal `"app"`.
2. `grandes_surco`: las ventas de Surco con 5 unidades o más.
3. `jean_o_casaca`: las ventas de categoría `"jean"` o `"casaca"`, usando `isin`.
4. `caras_tienda`: las ventas con precio mayor que 150 hechas en el canal `"tienda"`, usando `query`.
5. `sin_resultados`: las ventas con más de 100 unidades (¿cuántas hay?).
6. `productos_caros`: solo la columna producto de las ventas con precio mayor que 150, con `loc`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

Cada punto es una línea. En los puntos 1 a 3 y 5 el patrón es `ventas[condición]`.
</details>

<details><summary>💡 Pista 2</summary>

En `query`, el texto del canal va entre comillas simples: `"precio > 150 and canal == 'tienda'"`. Para `productos_caros`: `ventas.loc[condición, "producto"]`.
</details>

---
## 🏋️ Reto final: explorar los movimientos
Con `movs`:
1. `n_clientes`: cuántos clientes distintos hay (investiga el método `nunique`).
2. `retiros_app`: los movimientos de tipo `"retiro"` hechos por `"app"`, con `query`.
3. `total_retiros_app`: la suma del monto de esos retiros, redondeada a 2 decimales.
4. `mov_mayor`: la fila completa del movimiento de mayor monto (usa `idxmax` y `loc`).
5. `clientes_grandes`: una **lista ordenada** de los clientes que hicieron algún depósito mayor que 2000, sin repetir.
6. `pct_negativos`: el porcentaje de movimientos con monto negativo, con 1 decimal.
7. `resumen_monto`: el resumen estadístico de la columna monto.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

`idxmax` devuelve la etiqueta de la fila, que luego usas en `loc`. Para el porcentaje, la media de una máscara es una proporción.
</details>

<details><summary>💡 Pista 2</summary>

Para `clientes_grandes`: filtra con `loc` las filas que cumplen, toma la columna cliente, quédate con los valores distintos (`.unique()`) y ordénalos con `sorted(...)`.
</details>

---
## 🚀 Nivel pro (opcional)
1. `mini`: carga desde `ventas.csv` solo las columnas fecha, tienda y unidades, y solo las 10 primeras filas. Investiga los parámetros `usecols` y `nrows` de `read_csv`.
2. `por_id`: `ventas` con la columna id_venta como índice (investiga `set_index`), y `venta_1025`: la fila de la venta 1025 buscada con `loc`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P2 (parte 1) · Cargar el dataset con pandas

**Qué hacer**
1. En `notebooks/02_limpieza_unificacion.ipynb`, carga con `pd.read_csv` el archivo (o los archivos) de la fuente que elegiste en P0. Si la fuente publica un archivo por año, carga cada uno en su propio DataFrame.
2. Revisa los parámetros de lectura hasta que los datos se vean bien: `sep` si las columnas salen todas juntas, `encoding` si aparecen caracteres raros en tildes y eñes (prueba `"utf-8"` y `"latin-1"`), y `skiprows` si el archivo trae filas de título antes del encabezado. Si solo tienes la versión en Excel, investiga `pd.read_excel`.
3. Explora cada archivo con `shape`, `head`, `info` y `describe`.
4. Si hay varios años, compara sus columnas (`list(df.columns)`) y anota las diferencias: nombres que cambian, columnas que aparecen o desaparecen. En la sesión 12 aprenderás a unirlos.
5. Actualiza el diccionario de datos de `data/README.md` con el tipo que pandas asignó a cada columna y marca las que tienen un tipo incorrecto: números o fechas leídos como texto, códigos leídos como números.
6. Marca las columnas con datos personales: se eliminarán en la limpieza.

**Por qué lo haría un analista**
La forma de leer el archivo decide todo lo que viene después: un separador o una codificación equivocados producen columnas mezcladas y textos ilegibles que parecen errores del análisis. Comparar las columnas entre años antes de unir evita perder datos o mezclar columnas que se llaman igual pero significan cosas distintas.

**Cómo debe verse el resultado**
Un notebook que carga todos los archivos sin errores ni caracteres raros, con una celda de texto al final que resume: filas y columnas de cada archivo, las columnas comunes y las diferentes entre años, y la lista de columnas cuyo tipo habrá que corregir en la sesión 10.

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Crear una Series con etiquetas y explicar qué pasa al sumar dos Series con etiquetas distintas.
- [ ] Crear un DataFrame desde un diccionario.
- [ ] Cargar un CSV, también con otro separador.
- [ ] Explorar un dataset con `head`, `shape`, `dtypes`, `info` y `describe`.
- [ ] Explicar por qué `df["col"]` y `df[["col"]]` devuelven tipos distintos.
- [ ] Explicar la diferencia entre `loc` e `iloc`, también en el slicing.
- [ ] Filtrar filas con varias condiciones, con `isin` y con `query`.

**Próxima sesión (S10):** limpieza de datos: nulos, duplicados, tipos, textos y columnas nuevas.